In [1]:
import pandas as pd

In [7]:
import sys
import numpy as np
import pandas as pd
import glob
from pathlib import Path
from scipy.stats import spearmanr

sys.path.insert(0, str(Path("/Users/narayanipemmaraju/Documents/MSDS/Spring 2026/ML/Project/Repo/StockTwit_WM/baselines")))

DAY_DIR = Path("/Users/narayanipemmaraju/Documents/MSDS/Spring 2026/ML/Project/Repo/StockTwit_WM/data/processed_day/by_split_month")

# ── load all splits ──────────────────────────────────────────────
train_day = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(str(DAY_DIR / "train/*.parquet")))], ignore_index=True)
val_day   = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(str(DAY_DIR / "val/*.parquet")))],   ignore_index=True)
test1_day = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(str(DAY_DIR / "test1/*.parquet")))], ignore_index=True)
test2_day = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(str(DAY_DIR / "test2/*.parquet")))], ignore_index=True)

# ── get day lists ────────────────────────────────────────────────
train_days = sorted(train_day['day'].unique())
val_days   = sorted(val_day['day'].unique())
test1_days = sorted(test1_day['day'].unique())
test2_days = sorted(test2_day['day'].unique())

print(f"Train: {len(train_days)} days | {min(train_days)} → {max(train_days)}")
print(f"Val:   {len(val_days)} days | {min(val_days)} → {max(val_days)}")
print(f"Test1: {len(test1_days)} days | {min(test1_days)} → {max(test1_days)}")
print(f"Test2: {len(test2_days)} days | {min(test2_days)} → {max(test2_days)}")

# ── build common roster ──────────────────────────────────────────
K = 100
roster_day = (
    train_day.groupby('symbol')['msg_count']
    .sum()
    .sort_values(ascending=False)
    .head(K)
    .index.tolist()
)

common_tickers_day = (
    set(roster_day)
    & set(train_day['symbol'].unique())
    & set(val_day['symbol'].unique())
    & set(test1_day['symbol'].unique())
    & set(test2_day['symbol'].unique())
)
roster_common_day = [t for t in roster_day if t in common_tickers_day]
print(f"\nRoster: {len(roster_day)} → Common: {len(roster_common_day)}")

# ── splits to evaluate on ────────────────────────────────────────
splits = [
    ("Val (2019)",    val_day,   val_days),
    ("Test1 (COVID)", test1_day, test1_days),
    ("Test2 (GME)",   test2_day, test2_days)
]

Train: 3870 days | 2008-05-27 00:00:00 → 2018-12-31 00:00:00
Val:   365 days | 2019-01-01 00:00:00 → 2019-12-31 00:00:00
Test1: 182 days | 2020-01-01 00:00:00 → 2020-06-30 00:00:00
Test2: 273 days | 2020-10-01 00:00:00 → 2021-06-30 00:00:00

Roster: 100 → Common: 79


In [8]:
from arima import PerTickerARIMA

# ── fit ──────────────────────────────────────────────────────────
arima_day = PerTickerARIMA(order=(2, 0, 1))
arima_day.fit(
    train_day, roster_common_day,
    log_attn_col='log_attention',
    week_col='day',
    symbol_col='symbol'
)

forecasts_day = arima_day.forecast(steps=13)

# ── evaluate on all splits ───────────────────────────────────────
for split_name, split_df, split_days in splits:
    print(f"\n── {split_name} ──")
    for h in [1, 4, 13]:
        if h > len(split_days):
            continue
        target_day = split_days[h - 1]
        actual = (
            split_df[split_df['day'] == target_day]
            .groupby('symbol')['log_attention']
            .first()
        )
        preds, actuals = [], []
        for ticker in roster_common_day:
            if ticker in actual.index and ticker in forecasts_day:
                fc = forecasts_day[ticker]
                fc_val = fc.iloc[h-1] if hasattr(fc, 'iloc') else (fc[h-1] if fc.ndim == 1 else fc[h-1, 0])
                preds.append(float(fc_val))
                actuals.append(float(actual[ticker]))
        mse = np.mean((np.array(preds) - np.array(actuals)) ** 2)
        mae = np.mean(np.abs(np.array(preds) - np.array(actuals)))
        rho, _ = spearmanr(preds, actuals)
        print(f"Horizon {h:2d} days → MSE: {mse:.4f} | MAE: {mae:.4f} | Spearman ρ: {rho:.4f}")

ARIMA fit: 100%|██████████| 79/79 [00:26<00:00,  2.95it/s]



── Val (2019) ──
Horizon  1 days → MSE: 1.2346 | MAE: 0.9609 | Spearman ρ: 0.7290
Horizon  4 days → MSE: 1.1517 | MAE: 0.8897 | Spearman ρ: 0.5677
Horizon 13 days → MSE: 1.2042 | MAE: 0.8777 | Spearman ρ: 0.2487

── Test1 (COVID) ──
Horizon  1 days → MSE: 1.7123 | MAE: 1.1358 | Spearman ρ: 0.4478
Horizon  4 days → MSE: 1.5292 | MAE: 1.0092 | Spearman ρ: 0.3691
Horizon 13 days → MSE: 1.6844 | MAE: 1.0558 | Spearman ρ: 0.2290

── Test2 (GME) ──
Horizon  1 days → MSE: 1.2289 | MAE: 0.8536 | Spearman ρ: 0.4922
Horizon  4 days → MSE: 1.0962 | MAE: 0.8844 | Spearman ρ: 0.3318
Horizon 13 days → MSE: 2.2204 | MAE: 1.1910 | Spearman ρ: 0.3148


In [9]:
from var import ReducedRankVAR

# ── fit ──────────────────────────────────────────────────────────
max_estimable = int((len(train_days) - 1) / len(roster_common_day)) - 1
maxlags = max(1, min(4, max_estimable))
print(f"Using maxlags={maxlags}")

var_day = ReducedRankVAR(maxlags=maxlags, rank=10)
var_day.fit(train_day, roster_common_day, log_attn_col='log_attention', week_col='day', symbol_col='symbol')

# ── build last_obs ───────────────────────────────────────────────
matrix_train_day = train_day[train_day['symbol'].isin(roster_common_day)].pivot_table(
    index='day', columns='symbol', values='log_attention', fill_value=0.0
).sort_index()[roster_common_day].values

last_obs = matrix_train_day[-var_day.result.k_ar:]

# ── forecast with fix ────────────────────────────────────────────
def var_forecast_fixed(var_model, last_obs, steps):
    p = var_model._coefs.shape[0]
    K = len(var_model.tickers)
    history = last_obs[-p:].copy()
    preds = []
    for _ in range(steps):
        y_hat = np.zeros(K)
        for l in range(p):
            y_hat += history[-(l + 1)] @ var_model._coefs[l].T
        preds.append(y_hat)
        history = np.vstack([history[1:], y_hat])
    return np.stack(preds)

forecasts_var_day = var_forecast_fixed(var_day, last_obs, steps=13)

# ── evaluate on all splits ───────────────────────────────────────
for split_name, split_df, split_days in splits:
    print(f"\n── {split_name} ──")
    matrix_split = split_df[split_df['symbol'].isin(roster_common_day)].pivot_table(
        index='day', columns='symbol', values='log_attention', fill_value=0.0
    ).sort_index()[roster_common_day].values

    for h in [1, 4, 13]:
        if h > len(split_days):
            continue
        predicted = forecasts_var_day[h - 1]
        actual    = matrix_split[h - 1]
        mse = np.mean((predicted - actual) ** 2)
        mae = np.mean(np.abs(predicted - actual))
        rho, _ = spearmanr(predicted, actual)
        print(f"Horizon {h:2d} days → MSE: {mse:.4f} | MAE: {mae:.4f} | Spearman ρ: {rho:.4f}")

Using maxlags=4
[VAR] fitted lag=4, rank=10, K=79

── Val (2019) ──
Horizon  1 days → MSE: 4.0805 | MAE: 1.5570 | Spearman ρ: 0.5177
Horizon  4 days → MSE: 7.7180 | MAE: 2.3696 | Spearman ρ: 0.5011
Horizon 13 days → MSE: 13.8469 | MAE: 3.3190 | Spearman ρ: 0.3586

── Test1 (COVID) ──
Horizon  1 days → MSE: 5.7274 | MAE: 1.8671 | Spearman ρ: 0.3331
Horizon  4 days → MSE: 4.3939 | MAE: 1.6674 | Spearman ρ: 0.3643
Horizon 13 days → MSE: 18.7592 | MAE: 3.5930 | Spearman ρ: 0.2452

── Test2 (GME) ──
Horizon  1 days → MSE: 11.0210 | MAE: 2.7392 | Spearman ρ: 0.2405
Horizon  4 days → MSE: 6.9443 | MAE: 2.0326 | Spearman ρ: 0.2277
Horizon 13 days → MSE: 16.8549 | MAE: 3.2179 | Spearman ρ: 0.0973


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from lstm import SharedLSTM, predict_lstm

# ── device ───────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ── build tensors ────────────────────────────────────────────────
FEATURES = ['log_attention', 'bullish_rate', 'bearish_rate', 'unlabeled_rate', 'attn_growth']

def build_tensor(panel_split, roster, features):
    matrices = []
    for feat in features:
        m = panel_split[panel_split['symbol'].isin(roster)].pivot_table(
            index='day', columns='symbol', values=feat, fill_value=0.0
        ).sort_index()[roster]
        matrices.append(m.values)
    return torch.tensor(np.stack(matrices, axis=-1).astype(np.float32))

train_tensor = build_tensor(train_day, roster_common_day, FEATURES)
val_tensor   = build_tensor(val_day,   roster_common_day, FEATURES)
test1_tensor = build_tensor(test1_day, roster_common_day, FEATURES)
test2_tensor = build_tensor(test2_day, roster_common_day, FEATURES)

print(f"Train: {train_tensor.shape}")
print(f"Val:   {val_tensor.shape}")
print(f"Test1: {test1_tensor.shape}")
print(f"Test2: {test2_tensor.shape}")

# ── build sequences ──────────────────────────────────────────────
SEQ_LEN = 20  # 20 days lookback (~1 month)
X, y = [], []
for i in range(len(train_tensor) - SEQ_LEN):
    X.append(train_tensor[i:i+SEQ_LEN])
    y.append(train_tensor[i+SEQ_LEN])

loader = DataLoader(
    TensorDataset(torch.stack(X), torch.stack(y)),
    batch_size=32, shuffle=True
)

# ── init model ───────────────────────────────────────────────────
K_common = len(roster_common_day)
lstm_day = SharedLSTM(n_tickers=K_common, feature_dim=5, hidden_dim=512, n_layers=2)
lstm_day = lstm_day.to(device)
optimizer = torch.optim.Adam(lstm_day.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)

# ── train 150 epochs ─────────────────────────────────────────────
best_val_loss = float('inf')
for epoch in range(150):
    # train
    lstm_day.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        pred = lstm_day(X_batch)[:, -1]
        loss = nn.functional.mse_loss(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_day.parameters(), 10.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # validation loss every 10 epochs
    if (epoch + 1) % 10 == 0:
        lstm_day.eval()
        with torch.no_grad():
            # build val sequences
            X_val, y_val = [], []
            for i in range(len(val_tensor) - SEQ_LEN):
                X_val.append(val_tensor[i:i+SEQ_LEN])
                y_val.append(val_tensor[i+SEQ_LEN])
            X_val = torch.stack(X_val).to(device)
            y_val = torch.stack(y_val).to(device)
            val_pred = lstm_day(X_val)[:, -1]
            val_loss = nn.functional.mse_loss(val_pred, y_val).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(lstm_day.state_dict(), "lstm_day_best.pt")

        print(f"Epoch {epoch+1}/150 — Train: {total_loss/len(loader):.4f} | Val: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# load best model
lstm_day.load_state_dict(torch.load("lstm_day_best.pt"))
print(f"\nLoaded best model (val loss: {best_val_loss:.4f})")

# ── evaluate on all splits ───────────────────────────────────────
context = train_tensor[-SEQ_LEN:]

for split_name, split_tensor, split_days in [
    ("Val (2019)",    val_tensor,   val_days),
    ("Test1 (COVID)", test1_tensor, test1_days),
    ("Test2 (GME)",   test2_tensor, test2_days)
]:
    print(f"\n── {split_name} ──")
    preds_lstm = predict_lstm(lstm_day, context, steps=13, device=device)
    for h in [1, 4, 13]:
        if h > len(split_days):
            continue
        predicted = preds_lstm[h-1, :, 0]
        actual    = split_tensor[h-1, :, 0].numpy()
        mse = float(np.mean((predicted - actual) ** 2))
        mae = float(np.mean(np.abs(predicted - actual)))
        rho, _ = spearmanr(predicted, actual)
        print(f"Horizon {h:2d} days → MSE: {mse:.4f} | MAE: {mae:.4f} | Spearman ρ: {float(rho):.4f}")

torch.save(lstm_day.state_dict(), "lstm_day_final.pt")
print("\nSaved lstm_day_final.pt ✓")

Using device: mps
Train: torch.Size([3853, 79, 5])
Val:   torch.Size([365, 79, 5])
Test1: torch.Size([182, 79, 5])
Test2: torch.Size([273, 79, 5])
Epoch 10/150 — Train: 0.1722 | Val: 0.7725 | LR: 0.000297
Epoch 20/150 — Train: 0.1395 | Val: 0.7006 | LR: 0.000287
Epoch 30/150 — Train: 0.1173 | Val: 0.6569 | LR: 0.000271
Epoch 40/150 — Train: 0.0988 | Val: 0.6660 | LR: 0.000250
Epoch 50/150 — Train: 0.0845 | Val: 0.7206 | LR: 0.000225
Epoch 60/150 — Train: 0.0733 | Val: 0.7319 | LR: 0.000196
Epoch 70/150 — Train: 0.0650 | Val: 0.7661 | LR: 0.000166
Epoch 80/150 — Train: 0.0592 | Val: 0.7659 | LR: 0.000134
Epoch 90/150 — Train: 0.0546 | Val: 0.7866 | LR: 0.000104
Epoch 100/150 — Train: 0.0515 | Val: 0.7797 | LR: 0.000075
Epoch 110/150 — Train: 0.0492 | Val: 0.7774 | LR: 0.000050
Epoch 120/150 — Train: 0.0476 | Val: 0.8003 | LR: 0.000029
Epoch 130/150 — Train: 0.0467 | Val: 0.7989 | LR: 0.000013
Epoch 140/150 — Train: 0.0462 | Val: 0.7945 | LR: 0.000003
Epoch 150/150 — Train: 0.0461 | Val:

New Metrics Included


In [12]:
from arima import PerTickerARIMA
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
import numpy as np

VIRALITY_K       = 20
VIRALITY_HORIZON = 4

arima_day = PerTickerARIMA(order=(2, 0, 1))
arima_day.fit(
    train_day, roster_common_day,
    log_attn_col='log_attention',
    week_col='day',
    symbol_col='symbol'
)
forecasts_day = arima_day.forecast(steps=13)

for split_name, split_df, split_days in splits:
    print(f"\n── {split_name} ──")
    for h in [1, 4, 13]:
        if h > len(split_days):
            continue
        target_day = split_days[h - 1]

        actual = (
            split_df[split_df['day'] == target_day]
            .groupby('symbol')['log_attention']
            .first()
        )

        # only use tickers present on this day
        valid_tickers = [t for t in roster_common_day 
                        if t in actual.index and t in forecasts_day]

        preds   = []
        actuals = []
        for t in valid_tickers:
            fc = forecasts_day[t]
            fc_val = fc.iloc[h-1] if hasattr(fc, 'iloc') else (fc[h-1] if fc.ndim == 1 else fc[h-1, 0])
            preds.append(float(fc_val))
            actuals.append(float(actual[t]))

        preds   = np.array(preds)
        actuals = np.array(actuals)

        # ── MSE, MAE, Spearman ──────────────────────────────────
        mse = np.mean((preds - actuals) ** 2)
        mae = np.mean(np.abs(preds - actuals))
        rho, _ = spearmanr(preds, actuals)

        # ── Precision@100 ───────────────────────────────────────
        k = min(100, len(valid_tickers))
        pred_top   = set(np.argsort(preds)[::-1][:k].tolist())
        actual_top = set(np.argsort(actuals)[::-1][:k].tolist())
        precision_at_100 = len(pred_top & actual_top) / k

        # ── AUC-ROC virality ────────────────────────────────────
        day_idx = split_days.index(target_day)
        auc_roc = float('nan')
        if day_idx + VIRALITY_HORIZON < len(split_days):
            future_days = split_days[day_idx + 1 : day_idx + VIRALITY_HORIZON + 1]
            future_df   = split_df[split_df['day'].isin(future_days)]

            viral_tickers = set()
            for fd in future_days:
                top20 = (
                    future_df[future_df['day'] == fd]
                    .groupby('symbol')['log_attention']
                    .first()
                    .nlargest(VIRALITY_K)
                    .index.tolist()
                )
                viral_tickers.update(top20)

            labels = np.array([1 if t in viral_tickers else 0 
                               for t in valid_tickers])
            scores = preds

            if labels.sum() > 0 and labels.sum() < len(labels):
                auc_roc = roc_auc_score(labels, scores)

        print(f"Horizon {h:2d} days → MSE: {mse:.4f} | MAE: {mae:.4f} | ρ: {rho:.4f} | P@100: {precision_at_100:.4f} | AUC: {auc_roc:.4f}")

ARIMA fit: 100%|██████████| 79/79 [00:30<00:00,  2.61it/s]



── Val (2019) ──
Horizon  1 days → MSE: 1.2346 | MAE: 0.9609 | ρ: 0.7290 | P@100: 1.0000 | AUC: 0.8468
Horizon  4 days → MSE: 1.1517 | MAE: 0.8897 | ρ: 0.5677 | P@100: 1.0000 | AUC: 0.7882
Horizon 13 days → MSE: 1.2042 | MAE: 0.8777 | ρ: 0.2487 | P@100: 1.0000 | AUC: 0.7513

── Test1 (COVID) ──
Horizon  1 days → MSE: 1.7123 | MAE: 1.1358 | ρ: 0.4478 | P@100: 1.0000 | AUC: 0.8344
Horizon  4 days → MSE: 1.5292 | MAE: 1.0092 | ρ: 0.3691 | P@100: 1.0000 | AUC: 0.8024
Horizon 13 days → MSE: 1.6844 | MAE: 1.0558 | ρ: 0.2290 | P@100: 1.0000 | AUC: 0.6846

── Test2 (GME) ──
Horizon  1 days → MSE: 1.2289 | MAE: 0.8536 | ρ: 0.4922 | P@100: 1.0000 | AUC: 0.7808
Horizon  4 days → MSE: 1.0962 | MAE: 0.8844 | ρ: 0.3318 | P@100: 1.0000 | AUC: 0.7407
Horizon 13 days → MSE: 2.2204 | MAE: 1.1910 | ρ: 0.3148 | P@100: 1.0000 | AUC: 0.7385


In [13]:
from var import ReducedRankVAR

max_estimable = int((len(train_days) - 1) / len(roster_common_day)) - 1
maxlags = max(1, min(4, max_estimable))
print(f"Using maxlags={maxlags}")

var_day = ReducedRankVAR(maxlags=maxlags, rank=10)
var_day.fit(train_day, roster_common_day, log_attn_col='log_attention', week_col='day', symbol_col='symbol')

matrix_train_day = train_day[train_day['symbol'].isin(roster_common_day)].pivot_table(
    index='day', columns='symbol', values='log_attention', fill_value=0.0
).sort_index()[roster_common_day].values

last_obs = matrix_train_day[-var_day.result.k_ar:]

def var_forecast_fixed(var_model, last_obs, steps):
    p = var_model._coefs.shape[0]
    K = len(var_model.tickers)
    history = last_obs[-p:].copy()
    preds = []
    for _ in range(steps):
        y_hat = np.zeros(K)
        for l in range(p):
            y_hat += history[-(l + 1)] @ var_model._coefs[l].T
        preds.append(y_hat)
        history = np.vstack([history[1:], y_hat])
    return np.stack(preds)

forecasts_var_day = var_forecast_fixed(var_day, last_obs, steps=13)

VIRALITY_K       = 20
VIRALITY_HORIZON = 4

for split_name, split_df, split_days in splits:
    print(f"\n── {split_name} ──")

    matrix_split = split_df[split_df['symbol'].isin(roster_common_day)].pivot_table(
        index='day', columns='symbol', values='log_attention', fill_value=0.0
    ).sort_index()[roster_common_day].values

    for h in [1, 4, 13]:
        if h > len(split_days):
            continue

        predicted = forecasts_var_day[h - 1]   # (K,)
        actual    = matrix_split[h - 1]         # (K,)

        # find valid tickers (non-zero actual — present that day)
        valid_idx = np.where(actual > 0)[0]
        if len(valid_idx) == 0:
            continue

        pred_valid   = predicted[valid_idx]
        actual_valid = actual[valid_idx]

        # ── MSE, MAE, Spearman ──────────────────────────────────
        mse = np.mean((pred_valid - actual_valid) ** 2)
        mae = np.mean(np.abs(pred_valid - actual_valid))
        rho, _ = spearmanr(pred_valid, actual_valid)

        # ── Precision@100 ───────────────────────────────────────
        k = min(100, len(valid_idx))
        pred_top   = set(np.argsort(pred_valid)[::-1][:k].tolist())
        actual_top = set(np.argsort(actual_valid)[::-1][:k].tolist())
        precision_at_100 = len(pred_top & actual_top) / k

        # ── AUC-ROC virality ────────────────────────────────────
        day_idx = h - 1
        auc_roc = float('nan')
        if day_idx + VIRALITY_HORIZON < len(matrix_split):
            future = matrix_split[day_idx + 1 : day_idx + VIRALITY_HORIZON + 1]
            viral_set = set()
            for row in future:
                viral_set.update(np.argsort(row)[::-1][:VIRALITY_K].tolist())

            labels = np.array([1 if i in viral_set else 0 for i in valid_idx])
            scores = pred_valid

            if labels.sum() > 0 and labels.sum() < len(labels):
                auc_roc = roc_auc_score(labels, scores)

        print(f"Horizon {h:2d} days → MSE: {mse:.4f} | MAE: {mae:.4f} | ρ: {rho:.4f} | P@100: {precision_at_100:.4f} | AUC: {auc_roc:.4f}")

Using maxlags=4
[VAR] fitted lag=4, rank=10, K=79

── Val (2019) ──
Horizon  1 days → MSE: 3.6852 | MAE: 1.4574 | ρ: 0.4032 | P@100: 1.0000 | AUC: 0.6089
Horizon  4 days → MSE: 9.1177 | MAE: 2.6460 | ρ: 0.2691 | P@100: 1.0000 | AUC: 0.5683
Horizon 13 days → MSE: 16.0271 | MAE: 3.6929 | ρ: 0.2660 | P@100: 1.0000 | AUC: 0.6071

── Test1 (COVID) ──
Horizon  1 days → MSE: 3.5945 | MAE: 1.4914 | ρ: 0.1408 | P@100: 1.0000 | AUC: 0.6083
Horizon  4 days → MSE: 4.1401 | MAE: 1.5919 | ρ: 0.2516 | P@100: 1.0000 | AUC: 0.5878
Horizon 13 days → MSE: 28.7007 | MAE: 4.9965 | ρ: -0.1444 | P@100: 1.0000 | AUC: 0.4371

── Test2 (GME) ──
Horizon  1 days → MSE: 5.2325 | MAE: 1.7490 | ρ: -0.0108 | P@100: 1.0000 | AUC: 0.4340
Horizon  4 days → MSE: 5.2769 | MAE: 1.7352 | ρ: 0.0030 | P@100: 1.0000 | AUC: 0.5263
Horizon 13 days → MSE: 33.4206 | MAE: 5.4692 | ρ: 0.1153 | P@100: 1.0000 | AUC: 0.3731


In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from lstm import SharedLSTM, predict_lstm

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

FEATURES = ['log_attention', 'bullish_rate', 'bearish_rate', 'unlabeled_rate', 'attn_growth']

def build_tensor(panel_split, roster, features):
    matrices = []
    for feat in features:
        m = panel_split[panel_split['symbol'].isin(roster)].pivot_table(
            index='day', columns='symbol', values=feat, fill_value=0.0
        ).sort_index()[roster]
        matrices.append(m.values)
    return torch.tensor(np.stack(matrices, axis=-1).astype(np.float32))

train_tensor = build_tensor(train_day, roster_common_day, FEATURES)
val_tensor   = build_tensor(val_day,   roster_common_day, FEATURES)
test1_tensor = build_tensor(test1_day, roster_common_day, FEATURES)
test2_tensor = build_tensor(test2_day, roster_common_day, FEATURES)

# ── build sequences ──────────────────────────────────────────────
SEQ_LEN = 20
X, y = [], []
for i in range(len(train_tensor) - SEQ_LEN):
    X.append(train_tensor[i:i+SEQ_LEN])
    y.append(train_tensor[i+SEQ_LEN])

loader = DataLoader(
    TensorDataset(torch.stack(X), torch.stack(y)),
    batch_size=32, shuffle=True
)

# ── init + train ─────────────────────────────────────────────────
K_common = len(roster_common_day)
lstm_day = SharedLSTM(n_tickers=K_common, feature_dim=5, hidden_dim=512, n_layers=2)
lstm_day = lstm_day.to(device)
optimizer = torch.optim.Adam(lstm_day.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)

best_val_loss = float('inf')
for epoch in range(150):
    lstm_day.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        pred = lstm_day(X_batch)[:, -1]
        loss = nn.functional.mse_loss(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_day.parameters(), 10.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    if (epoch + 1) % 10 == 0:
        lstm_day.eval()
        with torch.no_grad():
            X_val, y_val = [], []
            for i in range(len(val_tensor) - SEQ_LEN):
                X_val.append(val_tensor[i:i+SEQ_LEN])
                y_val.append(val_tensor[i+SEQ_LEN])
            X_val = torch.stack(X_val).to(device)
            y_val = torch.stack(y_val).to(device)
            val_pred = lstm_day(X_val)[:, -1]
            val_loss = nn.functional.mse_loss(val_pred, y_val).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(lstm_day.state_dict(), "lstm_day_best.pt")

        print(f"Epoch {epoch+1}/150 — Train: {total_loss/len(loader):.4f} | Val: {val_loss:.4f}")

lstm_day.load_state_dict(torch.load("lstm_day_best.pt"))
print(f"\nLoaded best model (val loss: {best_val_loss:.4f})")

# ── evaluate ─────────────────────────────────────────────────────
context = train_tensor[-SEQ_LEN:]
VIRALITY_K       = 20
VIRALITY_HORIZON = 4

for split_name, split_tensor, split_days in [
    ("Val (2019)",    val_tensor,   val_days),
    ("Test1 (COVID)", test1_tensor, test1_days),
    ("Test2 (GME)",   test2_tensor, test2_days)
]:
    print(f"\n── {split_name} ──")
    preds_lstm = predict_lstm(lstm_day, context, steps=13, device=device)

    for h in [1, 4, 13]:
        if h > len(split_days):
            continue

        predicted = preds_lstm[h-1, :, 0]           # (K,)
        actual    = split_tensor[h-1, :, 0].numpy() # (K,)

        # find valid tickers — non-zero actual (present that day)
        valid_idx = np.where(actual > 0)[0]
        if len(valid_idx) == 0:
            continue

        pred_valid   = predicted[valid_idx]
        actual_valid = actual[valid_idx]

        # ── MSE, MAE, Spearman ──────────────────────────────────
        mse = float(np.mean((pred_valid - actual_valid) ** 2))
        mae = float(np.mean(np.abs(pred_valid - actual_valid)))
        rho, _ = spearmanr(pred_valid, actual_valid)

        # ── Precision@100 ───────────────────────────────────────
        k = min(100, len(valid_idx))
        pred_top   = set(np.argsort(pred_valid)[::-1][:k].tolist())
        actual_top = set(np.argsort(actual_valid)[::-1][:k].tolist())
        precision_at_100 = len(pred_top & actual_top) / k

        # ── AUC-ROC virality ────────────────────────────────────
        day_idx = h - 1
        auc_roc = float('nan')
        if day_idx + VIRALITY_HORIZON < split_tensor.shape[0]:
            future = split_tensor[day_idx+1 : day_idx+VIRALITY_HORIZON+1, :, 0].numpy()
            viral_set = set()
            for row in future:
                viral_set.update(np.argsort(row)[::-1][:VIRALITY_K].tolist())

            labels = np.array([1 if i in viral_set else 0 for i in valid_idx])
            scores = pred_valid

            if labels.sum() > 0 and labels.sum() < len(labels):
                auc_roc = roc_auc_score(labels, scores)

        print(f"Horizon {h:2d} days → MSE: {mse:.4f} | MAE: {mae:.4f} | ρ: {float(rho):.4f} | P@100: {precision_at_100:.4f} | AUC: {auc_roc:.4f}")

torch.save(lstm_day.state_dict(), "lstm_day_final.pt")
print("\nSaved lstm_day_final.pt ✓")

Using device: mps
Epoch 10/150 — Train: 0.1699 | Val: 0.6981
Epoch 20/150 — Train: 0.1371 | Val: 0.6315
Epoch 30/150 — Train: 0.1146 | Val: 0.6129
Epoch 40/150 — Train: 0.0961 | Val: 0.6967
Epoch 50/150 — Train: 0.0814 | Val: 0.7256
Epoch 60/150 — Train: 0.0705 | Val: 0.7656
Epoch 70/150 — Train: 0.0625 | Val: 0.8404
Epoch 80/150 — Train: 0.0567 | Val: 0.8387
Epoch 90/150 — Train: 0.0525 | Val: 0.8548
Epoch 100/150 — Train: 0.0494 | Val: 0.8499
Epoch 110/150 — Train: 0.0472 | Val: 0.8712
Epoch 120/150 — Train: 0.0457 | Val: 0.8822
Epoch 130/150 — Train: 0.0448 | Val: 0.8678
Epoch 140/150 — Train: 0.0444 | Val: 0.8670
Epoch 150/150 — Train: 0.0442 | Val: 0.8651

Loaded best model (val loss: 0.6129)

── Val (2019) ──
Horizon  1 days → MSE: 0.8874 | MAE: 0.7662 | ρ: 0.7240 | P@100: 1.0000 | AUC: 0.8293
Horizon  4 days → MSE: 1.3602 | MAE: 0.9454 | ρ: 0.6738 | P@100: 1.0000 | AUC: 0.7524
Horizon 13 days → MSE: 1.3000 | MAE: 0.8694 | ρ: 0.4729 | P@100: 1.0000 | AUC: 0.6342

── Test1 (COVID)